# Session 10: ETL Pipeline Design

**Course:** Python for Data Engineering  
**Phase 3:** Data Engineering Concepts

**What we'll cover:**
- ETL vs ELT
- Batch vs streaming
- Pipeline architecture patterns
- Building a structured ETL pipeline

**Data files:** `data/sales.csv`, `data/employees.csv`, `data/orders_with_dates.csv`

**Note:** This session is demo-heavy. Follow along by running each cell.

---

## 1. ETL vs ELT

| | ETL | ELT |
|---|-----|-----|
| **Transform where?** | Before loading (in Python/Spark) | After loading (in the warehouse) |
| **Good for** | Complex transformations, legacy systems | Cloud warehouses (BigQuery, Snowflake) |
| **Example** | Read CSV → clean in pandas → write to DB | Load raw CSV to BigQuery → transform with SQL |

In this course we focus on **ETL** — transform happens in Python before loading. But know that many modern pipelines use ELT where the warehouse does the heavy lifting.

## 2. Batch vs Streaming

| | Batch | Streaming |
|---|-------|----------|
| **When** | Scheduled (hourly, daily) | Continuous / real-time |
| **Data** | Files, DB dumps | Events, messages |
| **Tools** | Airflow, cron, pandas | Kafka, Flink, Spark Streaming |
| **Example** | Daily sales report | Live fraud detection |

We'll work with **batch** pipelines — that's 90% of real-world data engineering.

---

## 3. Pipeline Architecture

A well-structured pipeline has clear separation:

```
config.json → extract() → transform() → load() → summary/logs
```

Each step is a function. Each function does one job. Logging throughout. Error handling at each stage.

Let's build one step by step.

In [ ]:
import pandas as pd
import json
import logging
from datetime import datetime

In [ ]:
# Setup logging for the pipeline

log = logging.getLogger("etl_pipeline")
log.setLevel(logging.INFO)
log.handlers.clear()
handler = logging.StreamHandler()
handler.setFormatter(logging.Formatter("%(asctime)s | %(levelname)-8s | %(message)s", datefmt="%H:%M:%S"))
log.addHandler(handler)

### Step 1: Extract

Read data from the source. Could be CSV, JSON, API, database. The extract function should be generic and return raw data.

In [ ]:
def extract_csv(filepath):
    """Extract data from a CSV file."""
    log.info(f"Extracting from {filepath}")
    df = pd.read_csv(filepath)
    log.info(f"Extracted {len(df)} rows, {len(df.columns)} columns")
    return df

def extract_json(filepath):
    """Extract data from a JSON file."""
    log.info(f"Extracting from {filepath}")
    with open(filepath, "r") as f:
        data = json.load(f)
    df = pd.DataFrame(data)
    log.info(f"Extracted {len(df)} rows")
    return df

# Test
raw_sales = extract_csv("data/sales.csv")
raw_sales.head()

### Step 2: Transform

Clean, filter, enrich. This is where the business logic lives.

In [ ]:
def transform_sales(df):
    """Clean and enrich sales data."""
    log.info("Starting transform")
    initial_count = len(df)
    
    # Clean columns
    df = df.copy()
    df["product"] = df["product"].str.strip().str.title()
    df["region"] = df["region"].str.strip().str.lower()
    df["unit_price"] = df["unit_price"].str.replace("$", "", regex=False).astype(float)
    df["quantity"] = df["quantity"].astype(int)
    
    # Filter invalid rows
    invalid = df[df["quantity"] <= 0]
    if len(invalid) > 0:
        log.warning(f"Dropping {len(invalid)} rows with quantity <= 0")
    df = df[df["quantity"] > 0].copy()
    
    # Enrich
    df["total"] = (df["quantity"] * df["unit_price"]).round(2)
    df["date"] = pd.to_datetime(df["date"])
    
    log.info(f"Transform complete: {initial_count} → {len(df)} rows")
    return df

# Test
clean_sales = transform_sales(raw_sales)
clean_sales

### Step 3: Load

Write the clean data to the target — CSV, database, cloud storage, etc.

In [ ]:
def load_csv(df, filepath):
    """Load data to a CSV file."""
    log.info(f"Loading {len(df)} rows to {filepath}")
    df.to_csv(filepath, index=False)
    log.info("Load complete")

def load_json(df, filepath):
    """Load data to a JSON file."""
    log.info(f"Loading {len(df)} rows to {filepath}")
    df.to_json(filepath, orient="records", indent=2)
    log.info("Load complete")

# Test
load_csv(clean_sales, "data/sales_pipeline_output.csv")

### Step 4: Orchestrate — Tying it all together

A `run_pipeline()` function that calls extract → transform → load in sequence, with error handling and a summary.

In [ ]:
def run_pipeline(source_path, target_path, transform_fn):
    """Run a complete ETL pipeline."""
    start_time = datetime.now()
    log.info("=" * 50)
    log.info("PIPELINE STARTED")
    
    try:
        # Extract
        raw = extract_csv(source_path)
        
        # Transform
        clean = transform_fn(raw)
        
        # Load
        load_csv(clean, target_path)
        
        # Summary
        duration = (datetime.now() - start_time).total_seconds()
        summary = {
            "status": "success",
            "source": source_path,
            "target": target_path,
            "rows_in": len(raw),
            "rows_out": len(clean),
            "rows_dropped": len(raw) - len(clean),
            "duration_seconds": round(duration, 2),
            "timestamp": start_time.isoformat(),
        }
        log.info(f"PIPELINE COMPLETE in {duration:.2f}s")
        return summary
        
    except Exception as e:
        log.error(f"PIPELINE FAILED: {e}")
        return {"status": "failed", "error": str(e)}

# Run it
result = run_pipeline("data/sales.csv", "data/sales_clean_output.csv", transform_sales)
print("\nPipeline result:")
print(json.dumps(result, indent=2))

---

## 4. Config-Driven Pipelines

Hardcoding file paths is bad practice. Use a config file so you can change sources/targets without touching code.

In [ ]:
# Create a pipeline config
config = {
    "pipeline_name": "sales_etl",
    "source": {"type": "csv", "path": "data/sales.csv"},
    "target": {"type": "csv", "path": "data/sales_final.csv"},
    "settings": {
        "drop_invalid": True,
        "min_quantity": 1
    }
}

with open("data/etl_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Config saved")

In [ ]:
# Config-driven pipeline runner

def run_from_config(config_path):
    """Run pipeline based on a JSON config file."""
    # Load config
    with open(config_path, "r") as f:
        config = json.load(f)
    
    log.info(f"Running pipeline: {config['pipeline_name']}")
    
    # Extract
    source = config["source"]
    if source["type"] == "csv":
        raw = extract_csv(source["path"])
    elif source["type"] == "json":
        raw = extract_json(source["path"])
    else:
        raise ValueError(f"Unknown source type: {source['type']}")
    
    # Transform
    clean = transform_sales(raw)
    
    # Load
    target = config["target"]
    if target["type"] == "csv":
        load_csv(clean, target["path"])
    elif target["type"] == "json":
        load_json(clean, target["path"])
    
    log.info("Pipeline finished")
    return clean

result_df = run_from_config("data/etl_config.json")
result_df

---

## 5. Multi-Source Pipeline

Real pipelines often combine data from multiple sources. Here's a pattern for that.

In [ ]:
# Pipeline that combines sales and employee data into a report

def run_combined_pipeline():
    log.info("Starting combined pipeline")
    
    # Extract from multiple sources
    sales = extract_csv("data/orders_with_dates.csv")
    employees = extract_csv("data/employees.csv")
    
    # Transform sales
    sales["total"] = sales["quantity"] * sales["unit_price"]
    sales["order_timestamp"] = pd.to_datetime(sales["order_timestamp"])
    sales["month"] = sales["order_timestamp"].dt.to_period("M")
    
    # Transform employees
    employees["name"] = employees["name"].str.strip().str.title()
    employees["salary"] = employees["salary"].str.replace("$", "", regex=False).str.replace(",", "", regex=False).astype(float)
    employees["department"] = employees["department"].str.strip().str.lower()
    
    # Aggregate sales by region
    region_summary = sales.groupby("region").agg(
        total_orders=("order_id", "count"),
        total_revenue=("total", "sum"),
    ).reset_index()
    
    # Aggregate employees by department
    dept_summary = employees.groupby("department").agg(
        headcount=("name", "count"),
        total_salary=("salary", "sum"),
    ).reset_index()
    
    # Load outputs
    load_csv(region_summary, "data/region_summary.csv")
    load_csv(dept_summary, "data/dept_summary.csv")
    
    log.info("Combined pipeline complete")
    return region_summary, dept_summary

regions, depts = run_combined_pipeline()
print("\nRegion Summary:")
print(regions)
print("\nDepartment Summary:")
print(depts)

---

## Lab: Build a Complete ETL Pipeline

Build a pipeline from scratch for `data/orders_with_dates.csv`:

1. **Extract**: Read the CSV
2. **Transform**:
   - Convert timestamps
   - Add `total`, `quarter`, `month_name` columns
   - Categorize orders: `small` (< $100), `medium` ($100-500), `large` (> $500)
3. **Load**: Save enriched data to `data/orders_etl_output.csv`
4. **Report**: Save a JSON summary to `data/etl_report.json` with:
   - total_orders, total_revenue
   - revenue_by_quarter (dict)
   - top_customer (by total spent)
   - pipeline timestamp and duration
5. Add logging to every step

In [ ]:
import pandas as pd
import json
import logging
from datetime import datetime

# Your code here


---

## Summary

| Topic | Key Takeaway |
|-------|--------------|
| ETL vs ELT | ETL transforms in code, ELT transforms in the warehouse |
| Batch vs Streaming | Batch = scheduled runs, Streaming = continuous |
| Pipeline structure | `extract()` → `transform()` → `load()` as separate functions |
| Config-driven | Use JSON config files, not hardcoded paths |
| Logging | Log every step — it's your debugging lifeline in production |
| Summary/report | Always produce a run summary with row counts, duration, status |

**Key patterns:**
- Each step is a function with clear input/output
- Wrap the whole pipeline in try/except
- Config files make pipelines reusable across environments
- Always log: rows in, rows out, rows dropped, duration

**Next session:** Working with APIs — fetching data from REST APIs and storing it.